In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import BinaryEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn import set_config
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

#### 데이터 로딩

In [2]:
X_train = pd.read_csv('../data/X_train.csv', encoding='cp949')
y_train = pd.read_csv('../data/y_train.csv', encoding='cp949').Salary

In [3]:
# 결측치가 있는 피처
missing_features = X_train.columns[X_train.isnull().any()].tolist()
missing_features 

['ord_01', 'nom_04', 'nom_07']

In [4]:
X_train.head()

,ord_00,ord_01,nom_02,nom_03,nom_04,nom_05,nom_06,nom_07,bin_08,bin_09,...,bin_65,bin_66,bin_67,bin_68,bin_69,bin_70,bin_71,bin_72,bin_73,bin_74
0,beginner,C,jurl4c,j5y8n4ti0h,NaN,9w53,w8qwqpmd,NaN,Y,F,...,F,F,T,T,F,F,F,F,F,F
1,intermediate,NaN,rhxddv,7xoq0dv6xk,srdu,xqvc,bo0av3cm,kdzg,Y,F,...,F,F,F,T,F,F,F,F,F,F
2,beginner,D,kx7u0x,q51yptv6xe,NaN,3x38,njeztpq1,NaN,N,F,...,F,F,T,T,F,F,F,F,F,F
3,beginner,C,m4b0oq,tvnxakw782,srdu,3x38,uvp46vfv,NaN,N,F,...,F,F,T,T,F,F,F,F,F,F
4,beginner,C,xpu0c6,bopdwcoi9h,NaN,lutd,d9qo02zr,NaN,Y,T,...,F,F,F,F,F,F,F,F,F,F


In [5]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16570 entries, 0 to 16569
Data columns (total 75 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ord_00  16570 non-null  object
 1   ord_01  14600 non-null  object
 2   nom_02  16570 non-null  object
 3   nom_03  16570 non-null  object
 4   nom_04  6661 non-null   object
 5   nom_05  16570 non-null  object
 6   nom_06  16570 non-null  object
 7   nom_07  4988 non-null   object
 8   bin_08  16570 non-null  object
 9   bin_09  16570 non-null  object
 10  bin_10  16570 non-null  object
 11  bin_11  16570 non-null  object
 12  bin_12  16570 non-null  object
 13  bin_13  16570 non-null  object
 14  bin_14  16570 non-null  object
 15  bin_15  16570 non-null  object
 16  bin_16  16570 non-null  object
 17  bin_17  16570 non-null  object
 18  bin_18  16570 non-null  object
 19  bin_19  16570 non-null  object
 20  bin_20  16570 non-null  object
 21  bin_21  16570 non-null  object
 22  bin_22  16570 non-null

- binary가 압도적으로 많음

---

In [6]:
#score 데이터프레임 만들기
data = { "Model": [],
        "Encoder": [],
        "Mean Score": [],
        "Std Score": []
        }

score_df = pd.DataFrame(data)
score_df
score_df = pd.DataFrame(columns=["Model", "Encoder", "Mean Score", "Std Score"])

---

In [7]:
# RandomForestRegressor

preprocessor = Pipeline(
    steps=[ 
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder",  OneHotEncoder(sparse_output=True, handle_unknown='ignore'))
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor), 
        ( "regressor", RandomForestRegressor())
    ]
)

scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5)

# 변수 정의
name = type(model["regressor"]).__name__
encoder_name = model["preprocessor"]["encoder"].__class__.__name__
score_mean = round(np.sqrt(-1 * scores.mean()), 2)  
score_std = round(np.sqrt(scores.std()), 2) 

# score_df에 값 추가
score_df.loc[len(score_df)] = [name, encoder_name, score_mean, score_std]

# 성능 출력
print(f"{name} CV mean = {score_mean} with std = {score_std}")

KeyboardInterrupt: 

In [ ]:
# MLPRegressor
preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder",  OneHotEncoder(sparse_output=True, handle_unknown='ignore')),
    ]
)

model2_MLP = Pipeline(
    steps=[
        ("preprocessor", preprocessor), 
        ( "regressor", MLPRegressor())
    ]
)

name = type(model2_MLP["regressor"]).__name__
encoder_name = model2_MLP["preprocessor"]["encoder"].__class__.__name__  #추가
score_mean = round(np.sqrt(-1 * scores.mean()), 2)  #추가
score_std = round(np.sqrt(scores.std()), 2)   #추가

scores = cross_val_score(model2_MLP, X_train, y_train, scoring='neg_mean_squared_error', cv=5)


print(f"{name} CV mean = {score_mean} with std = {score_std}") #변수명으로만 바꿈

In [ ]:
score_df